# 02 Experiment Runner

Run only the report-facing experiments. The design is intentionally compact:

| Experiment | Backbone | Methods | Purpose |
|---|---|---|---|
| Main ablation | CLIP | baseline / query_decomp / bu_pg / full | Show module contributions in a reproduction-style setting. |
| Backbone replacement | SigLIP2 | baseline / full | Compare CLIP with the stronger SigLIP2 backbone. |
| Novel-location OOD-1 | CLIP and SigLIP2 | baseline / full | Test robustness after shifting the target moment by 10 seconds. |

Snippet-count sweeps and smoke/debug runs are intentionally excluded from the report experiment matrix.

## Setup

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from eval import parse_limit
from run_experiments import run_comparison

OUTPUT_EXPERIMENTS = PROJECT_ROOT / "outputs" / "experiments"
I3D_CHECKPOINT = PROJECT_ROOT / "outputs" / "models" / "i3d" / "rgb_imagenet.pt"

CLIP_MODEL = "openai/clip-vit-base-patch32"
SIGLIP2_MODEL = "google/siglip2-base-patch16-224"

METHOD_LABELS = {
    "baseline": "Baseline",
    "query_decomp": "Query Decomp.",
    "bu_pg": "w/o QC-FR",
    "full": "Full",
}
REPORT_METRICS = ["R@1_IoU_0.3", "R@1_IoU_0.5", "R@1_IoU_0.7", "mIoU"]

PROJECT_ROOT

## Configuration

In [ ]:
RUN_EXPERIMENTS = False

COMMON = {
    "limit": "full",
    "num_frames": 32,
    "snippet_frames": 16,
    "smooth_kernel": 5,
    "threshold_ratio": 0.75,
    "i3d_checkpoint": I3D_CHECKPOINT,
    "i3d_batch_size": 8,
    "alignment_image_batch_size": 64,
    "query_backend": "spacy",
    "qc_lambda": 0.5,
    "context_distance": 2,
    "proposal_k": 6,
    "min_proposal_snippets": 0,
    "proposal_method": "kmeans",
    "require_overlap": True,
}

EXPERIMENTS = {
    "main_ablation_clip": {
        "group": "main_ablation",
        "description": "Main module ablation with CLIP.",
        "methods": ["baseline", "query_decomp", "bu_pg", "full"],
        "alignment_model_name": CLIP_MODEL,
        "novel_location_prefix_seconds": 0.0,
        "novel_location_seed": 0,
    },
    "backbone_siglip2": {
        "group": "backbone_replacement",
        "description": "SigLIP2 counterpart for CLIP vs SigLIP2 comparison.",
        "methods": ["baseline", "full"],
        "alignment_model_name": SIGLIP2_MODEL,
        "novel_location_prefix_seconds": 0.0,
        "novel_location_seed": 0,
    },
    "ood1_clip": {
        "group": "novel_location_ood1",
        "description": "CLIP novel-location OOD-1 with a 10-second prefix.",
        "methods": ["baseline", "full"],
        "alignment_model_name": CLIP_MODEL,
        "novel_location_prefix_seconds": 10.0,
        "novel_location_seed": 0,
    },
    "ood1_siglip2": {
        "group": "novel_location_ood1",
        "description": "SigLIP2 novel-location OOD-1 with a 10-second prefix.",
        "methods": ["baseline", "full"],
        "alignment_model_name": SIGLIP2_MODEL,
        "novel_location_prefix_seconds": 10.0,
        "novel_location_seed": 0,
    },
}

pd.DataFrame.from_dict(EXPERIMENTS, orient="index")[["group", "description", "methods", "alignment_model_name", "novel_location_prefix_seconds"]]

## Run Experiments

In [ ]:
if RUN_EXPERIMENTS:
    for experiment_name, cfg in EXPERIMENTS.items():
        print(f"\n### Running {experiment_name}: {cfg['description']}")
        run_comparison(
            project_root=PROJECT_ROOT,
            methods=cfg["methods"],
            limit=parse_limit(str(COMMON["limit"])),
            num_frames=COMMON["num_frames"],
            snippet_frames=COMMON["snippet_frames"],
            smooth_kernel=COMMON["smooth_kernel"],
            threshold_ratio=COMMON["threshold_ratio"],
            alignment_model_name=cfg["alignment_model_name"],
            i3d_checkpoint=COMMON["i3d_checkpoint"],
            i3d_batch_size=COMMON["i3d_batch_size"],
            alignment_image_batch_size=COMMON["alignment_image_batch_size"],
            query_backend=COMMON["query_backend"],
            qc_lambda=COMMON["qc_lambda"],
            context_distance=COMMON["context_distance"],
            proposal_k=COMMON["proposal_k"],
            min_proposal_snippets=COMMON["min_proposal_snippets"],
            proposal_method=COMMON["proposal_method"],
            require_overlap=COMMON["require_overlap"],
            novel_location_prefix_seconds=cfg["novel_location_prefix_seconds"],
            novel_location_seed=cfg["novel_location_seed"],
            experiment_name=experiment_name,
        )
else:
    print("RUN_EXPERIMENTS is False. Existing metrics CSV files will be loaded below if present.")

## Load Compact Tables

In [ ]:
def load_metrics(experiment_names):
    rows = []
    missing = []
    for name in experiment_names:
        path = OUTPUT_EXPERIMENTS / f"{name}_metrics.csv"
        if not path.exists():
            missing.append(path)
            continue
        df = pd.read_csv(path)
        df.insert(0, "experiment", name)
        df.insert(1, "group", EXPERIMENTS[name]["group"])
        df.insert(2, "description", EXPERIMENTS[name]["description"])
        rows.append(df)
    if missing:
        print("Missing metrics files:")
        for path in missing:
            print(" -", path)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def compact_metrics(df):
    if df.empty:
        return df
    out = df[["group", "experiment", "method", "alignment_model_name", "novel_location_prefix_seconds"] + REPORT_METRICS].copy()
    out["method"] = out["method"].map(METHOD_LABELS).fillna(out["method"])
    out["backbone"] = out["alignment_model_name"].map({CLIP_MODEL: "CLIP", SIGLIP2_MODEL: "SigLIP2"}).fillna(out["alignment_model_name"])
    out[REPORT_METRICS] = out[REPORT_METRICS].round(4)
    return out[["group", "backbone", "method", "novel_location_prefix_seconds"] + REPORT_METRICS]

metrics = load_metrics(EXPERIMENTS.keys())
compact = compact_metrics(metrics)
compact

## Suggested PowerShell Commands

In [ ]:
python_exe = r"D:\Anaconda\envs\RP\python.exe"
base = f'{python_exe} .\src\run_experiments.py --limit full --num-frames 32 --snippet-frames 16 --query-backend spacy --i3d-batch-size 8 --alignment-image-batch-size 64 --min-proposal-snippets 0'
for experiment_name, cfg in EXPERIMENTS.items():
    methods = " ".join(cfg["methods"])
    command = (
        f'{base} --methods {methods} '
        f'--alignment-model-name {cfg["alignment_model_name"]} '
        f'--novel-location-prefix-seconds {cfg["novel_location_prefix_seconds"]:g} '
        f'--novel-location-seed {cfg["novel_location_seed"]} '
        f'--experiment-name {experiment_name}'
    )
    print(command)